# 08 — Character Lab: اختر، شاهد، ثبّت، درّب، اختبر
دفتر تفاعلي جديد: بوابة موافقة قبل الإنتاج، تدريب تركيبي للحركة، مقارنة baseline، وتدريب Qwen اختياري مع تقييم قبل/بعد. لا تُشغّل Run all لتجاوز الموافقة.

**حدود صريحة:** الحركة بيانات صناعية وليست motion capture. جودة كتابة السكربت تحتاج مراجعة بشرية. لا ندّعي تحسين التفكير الداخلي للمودل؛ نقيس التزامه بالمخطط وإصلاح الأخطاء. مسار Qwen يحتاج GPU وتنزيل أوزان من Hugging Face في بيئة تسمح بذلك. الفيديو هنا صامت؛ الصوت والمزامنة مرحلة مستقلة.

## 1. البيئة
افتح الدفتر من checkout المشروع. في Colab انسخ المشروع أولًا، ثم غيّر المسار. التثبيت اختياري وصريح.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json
ROOT = Path.cwd()
if not (ROOT / "voice2reel").is_dir() and (ROOT.parent / "voice2reel").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "voice2reel").is_dir(), "غيّر ROOT إلى مجلد المشروع"
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
INSTALL_CPU = False
if INSTALL_CPU:
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.2.6",
                    "Pillow==11.3.0", "imageio-ffmpeg==0.6.0", "ipywidgets>=8,<9",
                    "pydantic==2.11.7", "arabic-reshaper==3.0.0", "python-bidi==0.6.6"], check=True)
OUT = ROOT / "outputs/character_lab"
OUT.mkdir(parents=True, exist_ok=True)
from IPython.display import display, Video
from voice2reel.character_templates import (choice_list, render_preview, render_gallery,
                                           lock_template, load_lock)
print("Output:", OUT)

## 2. قائمة الأشكال ومعاينتها
اختر من القائمة ثم اضغط **معاينة**. لا يُثبّت أي شكل تلقائيًا. يمكنك أيضًا تغيير SELECTED وإعادة تشغيل الخلية التالية دون widgets.

In [ ]:
print(choice_list())
display(render_gallery(OUT / "catalog.png"))

In [ ]:
SELECTED = "classic"  # تغيير هذه القيمة ثم تشغيل الخلية يعرض صورة فقط
PREVIEWED = SELECTED
display(render_preview(SELECTED, output=OUT / "selected.png"))

In [ ]:
# واجهة list اختيارية. التأكيد يحدث فقط بعد المعاينة الحالية.
try:
    import ipywidgets as w
    from voice2reel.character_templates import TEMPLATE_IDS
    picker = w.Dropdown(options=list(TEMPLATE_IDS), value=SELECTED, description="الشكل")
    preview_button = w.Button(description="معاينة")
    confirm_button = w.Button(description="موافق: تثبيت الشكل")
    panel = w.Output()
    def preview_clicked(_):
        global SELECTED, PREVIEWED
        SELECTED = picker.value
        PREVIEWED = SELECTED
        with panel:
            panel.clear_output(wait=True)
            display(render_preview(SELECTED, output=OUT / "selected.png"))
    def confirm_clicked(_):
        with panel:
            if picker.value != PREVIEWED:
                print("اعرض معاينة الاختيار الحالي أولًا")
                return
            record = lock_template(PREVIEWED, OUT / "character.lock.json")
            print("تم تثبيت:", record["template_id"])
    preview_button.on_click(preview_clicked)
    confirm_button.on_click(confirm_clicked)
    display(w.VBox([picker, w.HBox([preview_button, confirm_button]), panel]))
except ImportError:
    print("استخدم خلية التأكيد اليدوي التالية أو ثبّت ipywidgets")

## 3. موافقة يدوية بديلة
إذا استخدمت الزر لا تحتاج تغيير هذه الخلية. خلاف ذلك اكتب نفس ID الذي شاهدته في APPROVE_ID. تركه فارغًا يمنع الاستمرار دون موافقة.

In [ ]:
APPROVE_ID = ""  # اكتب معرّف الشخصية التي توافق عليها بعد مشاهدة صورتها
if APPROVE_ID:
    assert APPROVE_ID == PREVIEWED, "اعرض معاينة هذا الشكل أولًا"
    lock_template(APPROVE_ID, OUT / "character.lock.json")
CHARACTER = load_lock(OUT / "character.lock.json")  # يتوقف هنا حتى توافق
assert CHARACTER["template_id"] == PREVIEWED, "الاختيار تغيّر؛ أكّد الشكل الجديد"
print("الشخصية المثبتة:", CHARACTER["template_id"])

## 4. تدريب حركة قابل للقياس — لا حفظ أسماء الحركات
720 خطة تدريب، 80 تحقق لعائلة nod، 80 اختبار لعائلة celebrate، و80 extrapolation بمعاملات أعلى. تتعلم شبكة خطية مع Adam تركيب ثماني بدائيات وحواف زمنية دورية. فصل العائلات يمنع تكرار العائلة لكنه لا يجعل البيانات واقعية: المعلم والمتعلم يشتركان في انحياز harmonics. المقارنة مع nearest-plan ومتوسط المسارات؛ لا نستخدم الاختبار لاختيار checkpoint.

In [ ]:
from voice2reel.motion_curriculum import build_curriculum, MotionLearner, FamilyMemoryBaseline
from voice2reel.common import write_json
splits = build_curriculum()
learner = MotionLearner(epochs=60).fit(splits["train"], splits["validation"])
nearest = FamilyMemoryBaseline(splits["train"])
scores = {name: {"learner_mae_px": learner.mean_error(plans),
                 "nearest_mae_px": nearest.mean_error(plans)}
          for name, plans in splits.items()}
write_json(OUT / "motion_scores.json", scores)
import numpy as np
np.savez(OUT / "motion_weights.npz", weights=learner.weights)
print(json.dumps(scores, indent=2))
assert scores["test"]["learner_mae_px"] < scores["test"]["nearest_mae_px"], "لم يتحسن؛ لا تنشر"

## 5. تجربة فشل مهمة: كشف عيوب لم يرها التدريب
التقرير يتضمن gate تجريبيًا يتعلم thresholds من عيوب محددة. دقته على flicker/abrupt المحجوبتين ضعيفة؛ لا تستخدمه كضمان جودة إنتاج. الإنتاج يستخدم فقط فحوص حدود المفاصل والقفزات مع مراجعة مرئية بشرية.

In [ ]:
from voice2reel.motion_curriculum import build_gate_corpus, MotionQualityGate
corpus = build_gate_corpus()
gate = MotionQualityGate().fit(corpus["train"])
print({name: gate.accuracy(rows) for name, rows in corpus.items()})

## 6. إخراج الحركة المتعلّمة بالشخصية المثبتة
MP4 عمودي 540×960، H.264 و24fps، بلا صوت. هذه دورة حركة واحدة خلال المدة المختارة، وليست مزامنة كلام. افحص الوجه، الأطراف والهوامش بصريًا قبل النشر.

In [ ]:
from voice2reel.character_studio import render_clip
assert load_lock(OUT / "character.lock.json")["template_id"] == CHARACTER["template_id"]
clip = render_clip(OUT / "character.lock.json", learner, splits["test"][0],
                   OUT / "motion_preview.mp4", seconds=4, fps=24)
display(Video(str(clip), embed=True))

## 7. Qwen: تدريب كتابة منظّمة + اختبار قبل/بعد
التدريب يستخدم مجموعة scripts فقط، لا يخلط Python عشوائيًا معها ولا ينفّذ كودًا مولّدًا. النموذج مثبت revision في وحدة train. التدريب masked-answer QLoRA مع اختيار أفضل checkpoint بخسارة validation. قياس المخطط لا يثبت تحسّن الحقيقة أو التفكير؛ أضف تقييمًا بشريًا أعمى للـhook والترابط والدقة والفائدة. اختبر القاعدة والـadapter على نفس العينة مرة واحدة بعد ضبط كل شيء على validation، واحفظ الردود الخام والأخطاء. **لم يُشغّل تدريب GPU في هذا التسليم.**

In [ ]:
RUN_QWEN_TRAIN = False
if RUN_QWEN_TRAIN:
    import torch
    assert torch.cuda.is_available(), "يحتاج GPU؛ ثبّت requirements/text.txt أولًا"
    subprocess.run([sys.executable, "-m", "voice2reel.scriptwriter.train",
                    "--epochs", "3", "--output", str(OUT / "qwen")], check=True)
else:
    print("Qwen training: NOT RUN")

## 8. قياس Qwen الخام ثم إصلاحه بملاحظات محددة
نقيس first-pass وrepair كلًا على حدة. لا نحسب JSON صالحًا على أنه سكربت ممتاز. 10 أمثلة اختبار بداية فقط؛ ارفع العدد لكل الاختبار لتقرير نهائي. هذه مرحلة inference-feedback وليست تدريب تفكير جديدًا.

In [ ]:
RUN_QWEN_EVAL = False
EVAL_LIMIT = 10
if RUN_QWEN_EVAL:
    from voice2reel.scriptwriter.generate import ScriptGenerator
    from voice2reel.scriptwriter.schema import messages_for, parse_script
    from voice2reel.character_studio import review_script
    from voice2reel.common import read_jsonl
    rows = read_jsonl(ROOT / "datasets/scripts/test.jsonl")[:EVAL_LIMIT]
    results = []
    for label, adapter in [("base", None), ("adapter", OUT / "qwen/adapter")]:
        generator = ScriptGenerator(adapter=adapter, device="cuda")
        for row in rows:
            expected = row["script"]
            messages = messages_for(row["idea"], expected["target_duration_seconds"], expected["language"])
            for attempt in range(2):
                raw = generator.complete(messages)
                record = {"model": label, "id": row["id"], "attempt": attempt, "raw": raw}
                try:
                    script = parse_script(raw, expected["target_duration_seconds"], expected["language"])
                    record["review"] = review_script(script)
                    record["valid"] = True
                except ValueError as exc:
                    record.update(valid=False, error=str(exc))
                results.append(record)
                if record["valid"]:
                    break
                messages += [{"role": "assistant", "content": raw},
                             {"role": "user", "content": "Correct only these validation errors: " + record["error"]}]
        del generator
        import gc
        import torch
        gc.collect()
        torch.cuda.empty_cache()
    write_json(OUT / "qwen_test_raw.json", results)
    for label in ("base", "adapter"):
        first = [r for r in results if r["model"] == label and r["attempt"] == 0]
        print(label, "first-pass valid:", sum(r["valid"] for r in first), "/", len(first))
else:
    print("Qwen evaluation: NOT RUN; no improvement claim")

## 9. قرار نشر بشري
- الشخصية مطابقة للمعاينة والموافقة؟
- لا انفصال أطراف، تقطيع، ومضات، أو خروج من الشاشة؟
- النص واضح، صحيح، غير مكرر، بلا ادعاءات غير موثقة؟
- عند إضافة صوت: أعد حساب المدة من waveform، ثم captions ثم mux؛ لا تفترض أن عدد الكلمات يضمن التزامن.
- احتفظ بـcharacter.lock.json وmotion_scores.json وMP4 sidecar وrun.json ونتائج Qwen الخام.

راجع `docs/CHARACTER_LAB.md` للنتائج الفعلية وحدودها. لا نعلن "قوي جدًا" اعتمادًا على loss وحده.